In [ ]:
# %% [markdown]
# # 강원도 전신주 산불 위험 예측 모델 (초안)
#
# 목표: 전신주(pole)마다 산불 위험 여부 `decision`(0/1)을 예측한다.
#
# 이 코드는 "완성본"이 아니라 회의용 **흐름 예시**다.
# 핵심은 다음 4단계 골격을 보여주는 것:
#   1) 전신주 좌표에 피처 붙이기 (지형 / 거리 / 날씨 / 토지피복)
#   2) 라벨(decision) 만들기  ← 회의에서 가장 많이 논의해야 할 부분
#   3) 모델 학습
#   4) 평가 + 전체 전신주 예측
#
# ※ 경로는 팀 폴더 구조에 맞춰 절대경로로 바꿔서 쓰는 걸 추천.

# %%
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 경로 (예시 - 실제 경로로 교체)
BASE = "data"
POLES_PATH   = f"{BASE}/예측데이터/gangwon_poles_4326.csv"
FIRE_PATH    = f"{BASE}/강원도_데이터/강원도_산불발생.csv"
FIRE_TERRAIN = f"{BASE}/강원도_데이터/산불_공간데이터/강원도_산불_지형특성계산.csv"
FIRE_STATION = f"{BASE}/강원도_데이터/산불_공간데이터/강원도_근방_소방서_위치.csv"
WEATHER_CELL = f"{BASE}/강원도_날씨데이터/강원도날씨_격자.csv"
WEATHER_DAY  = f"{BASE}/강원도_날씨데이터/강원도날씨_격자_일단위.csv"


# %% [markdown]
# ## 1단계. 데이터 로드

# %%
poles   = pd.read_csv(POLES_PATH)              # pole_id, lon, lat  (약 138만 행)
fire    = pd.read_csv(FIRE_PATH)               # 산불 발생 위치
terrain = pd.read_csv(FIRE_TERRAIN)            # 산불 지점의 지형특성
station = pd.read_csv(FIRE_STATION)            # 소방서 위치

# 회의 예시라 일부만 샘플링해서 빠르게 돌려봐도 됨 (실제론 전체)
# poles = poles.sample(50000, random_state=42).reset_index(drop=True)

print("전신주:", poles.shape)
print("산불:", fire.shape)


# %% [markdown]
# ## 2단계. 전신주에 피처 붙이기
#
# 전신주는 좌표만 있으므로, 다른 데이터를 공간적으로 연결(join)해서 피처를 만든다.
# 초안에서는 핵심 피처 3종만 붙인다.

# %%
# --- (a) 가장 가까운 "과거 산불 지점"의 지형값 가져오기 ---
# 산불이 났던 곳의 지형(고도/경사/TPI/TWI)을 전신주 주변 환경의 근사값으로 사용.
# (정석은 DEM에서 전신주 좌표 지형을 직접 추출하는 것 — 회의에서 논의 포인트)

fire_xy = terrain[["경도", "위도"]].to_numpy()
tree_fire = cKDTree(fire_xy)
pole_xy = poles[["lon", "lat"]].to_numpy()

dist, idx = tree_fire.query(pole_xy, k=1)   # 가장 가까운 산불 지점 1개
poles["nearest_fire_dist"] = dist           # 가장 가까운 산불까지 거리(도 단위, 임시)
poles["고도"]   = terrain["고도(m)"].to_numpy()[idx]
poles["경사도"] = terrain["경사도(도)"].to_numpy()[idx]
poles["TPI"]    = terrain["TPI(지형위치지수)"].to_numpy()[idx]
poles["TWI"]    = terrain["TWI(지형다습지수)"].to_numpy()[idx]

# --- (b) 가장 가까운 소방서까지 거리 ---
stn_xy = station[["경도", "위도"]].to_numpy()
tree_stn = cKDTree(stn_xy)
stn_dist, _ = tree_stn.query(pole_xy, k=1)
poles["소방서_거리"] = stn_dist

# --- (c) 날씨: 전신주가 속한 기상셀의 평균 날씨 ---
# 일단위 날씨를 셀별로 집계해서, 가까운 셀 중심값을 붙인다 (초안 버전).
cells = pd.read_csv(WEATHER_CELL)
wday  = pd.read_csv(WEATHER_DAY)

# 봄철(3~4월) 산불 위험기 평균만 예시로 사용
wday["월"] = pd.to_datetime(wday["날짜"]).dt.month
spring = wday[wday["월"].isin([3, 4])]
cell_w = spring.groupby("기상셀ID").agg(
    평균기온=("평균기온_C", "mean"),
    강수량=("강수량_mm", "mean"),
    최대풍속=("최대순간풍속_m_s", "mean"),
).reset_index()

cells = cells.merge(cell_w, on="기상셀ID", how="left")
cell_xy = cells[["중심경도_wgs84", "중심위도_wgs84"]].to_numpy()
tree_cell = cKDTree(cell_xy)
_, cidx = tree_cell.query(pole_xy, k=1)
poles["평균기온"] = cells["평균기온"].to_numpy()[cidx]
poles["봄_강수량"] = cells["강수량"].to_numpy()[cidx]
poles["봄_최대풍속"] = cells["최대풍속"].to_numpy()[cidx]

print(poles.head())


# %% [markdown]
# ## 3단계. 라벨(decision) 만들기  ★회의 핵심 논의 지점★
#
# 우리에겐 "이 전신주가 위험하다(1)"는 정답 라벨이 아직 없다.
# 그래서 직접 정의해야 한다. 아래는 가장 단순한 예시 규칙:
#   "과거 산불 지점에서 일정 거리(예: 1km) 안에 있는 전신주 = 위험(1)"
#
# → 거리 기준을 얼마로 할지, 지형/날씨 조건도 함께 넣을지는 회의에서 결정.

# %%
DIST_THRESHOLD = 0.01   # 약 1km (위경도 도 단위 근사). 회의에서 조정.
poles["decision"] = (poles["nearest_fire_dist"] < DIST_THRESHOLD).astype(int)

print(poles["decision"].value_counts())
# 보통 1(위험)이 훨씬 적은 불균형 데이터 → 회의에서 class_weight / 샘플링 논의


# %% [markdown]
# ## 4단계. 모델 학습 & 평가

# %%
features = ["고도", "경사도", "TPI", "TWI",
            "소방서_거리", "평균기온", "봄_강수량", "봄_최대풍속"]

X = poles[features].fillna(poles[features].median())
y = poles["decision"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",   # 불균형 보정
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

# 평가
pred  = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

# 어떤 피처가 중요했는지 (회의 발표용으로 좋음)
imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\n피처 중요도:\n", imp)


# %% [markdown]
# ## 5단계. 전체 전신주 예측 → 최종 산출물 형태

# %%
poles["pred_decision"] = model.predict(X)
poles["pred_proba"]    = model.predict_proba(X)[:, 1]

result = poles[["pole_id", "lon", "lat", "pred_decision", "pred_proba"]]
# result.to_csv("output/pole_decision_예측.csv", index=False)
print(result.head())


# %% [markdown]
# ## 회의에서 같이 정해야 할 것들 (TODO)
# - [ ] `decision` 라벨 정의 방법 (거리만? 지형/날씨 조건 추가?)
# - [ ] 전신주 지형값을 산불 근사 대신 DEM에서 직접 추출할지
# - [ ] 토지피복(gpkg) 피처 추가 (전신주가 침엽수림/도로 등 어디에 있는지)
# - [ ] 클래스 불균형 처리 (class_weight vs 언더/오버샘플링)
# - [ ] 모델 비교 (RandomForest vs XGBoost/LightGBM)
# - [ ] 거리 단위를 도(degree) 대신 미터(EPSG:5186 변환)로 통일